# German Credit Risk Analysis & Prediction

This notebook contains an end-to-end data analysis and machine learning classification pipeline utilizing the German Credit dataset.

In [ ]:
import pandas as pd
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'
columns = [
    'status_existing_checking_account', 'duration_months', 'credit_history',
    'purpose', 'credit_amount', 'savings_account_bonds', 'present_employment_since',
    'installment_rate_disposable_income', 'personal_status_sex', 'other_debtors_guarantors',
    'present_residence_since', 'property', 'age_years', 'other_installment_plans',
    'housing', 'number_credits_this_bank', 'job', 'number_people_liable_maintenance',
    'telephone', 'foreign_worker', 'credit_risk'
]
df_credit = pd.read_csv(url, sep=' ', names=columns)
df_credit.head()

In [ ]:
df_credit.info()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df_credit, x='age_years', kde=True, ax=axes[0], color='skyblue')
sns.countplot(data=df_credit, x='foreign_worker', hue='foreign_worker', ax=axes[1], palette='pastel', legend=False)
plt.tight_layout()
plt.show()

In [ ]:
mean_credit_by_risk = df_credit.groupby('credit_risk')['credit_amount'].mean().reset_index()
mean_credit_by_risk

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(data=df_credit, x='duration_months', y='credit_amount', scatter_kws={'alpha':0.5, 'color': 'teal'}, line_kws={'color': 'red'})
plt.show()

In [ ]:
job_map = {'A171': 'Unemployed/Unskilled Non-Res', 'A172': 'Unskilled Res', 'A173': 'Skilled Employee', 'A174': 'Management/Self-Emp'}
risk_map = {1: 'Low Risk', 2: 'High Risk'}
df_plot = df_credit.copy()
df_plot['Job Status'] = df_plot['job'].map(job_map)
df_plot['Credit Risk'] = df_plot['credit_risk'].map(risk_map)
plt.figure(figsize=(12, 6))
sns.countplot(data=df_plot, x='Job Status', hue='Credit Risk', palette='Set2')
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
df_ml = df_credit.copy()
for col in df_ml.select_dtypes(include=['object']).columns:
    df_ml[col] = LabelEncoder().fit_transform(df_ml[col])
X = df_ml.drop(columns=['credit_risk'])
y = df_ml['credit_risk']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)
print(classification_report(y_test, rf_model.predict(X_test)))